# 44 — Feature-family revalidation

Fair rolling comparison of `contract`, `contract + historical`, and `all` using the same algorithm, samples and training budget. Results are diagnostic and do not mutate ARGOS V1.


In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.robustness import load_robustness_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
(paths.root / "reports" / "robustness").mkdir(parents=True, exist_ok=True)
(paths.root / "data" / "m5").mkdir(parents=True, exist_ok=True)
(paths.root / "artifacts" / "models" / "m5").mkdir(parents=True, exist_ok=True)

m4_gate = paths.docs / "decisions" / "M4_GATE.md"
assert m4_gate.exists() and "M4 ✅ CLOSED" in m4_gate.read_text(encoding="utf-8"), (
    "M4 no está formalmente cerrado. No iniciar M5."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M5_SPEC = load_robustness_spec(paths.configs / "robustness.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

assert GOLD_SPEC.schema_version == M5_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M5_SPEC.core.benchmark_model_schema
{
    "gold_schema": GOLD_SPEC.schema_version,
    "model_schema": MODEL_SPEC.schema_version,
    "m5_schema": M5_SPEC.schema_version,
    "benchmark_model": M5_SPEC.core.benchmark_model_name,
    "benchmark_calibration": M5_SPEC.core.benchmark_calibration,
    "consumed_test_year": M5_SPEC.governance.consumed_test_year,
    "fresh_oot_year": M5_SPEC.governance.fresh_oot_year,
}

assert (paths.reports / "robustness" / "43_validation.parquet").exists(), "Ejecuta 43 primero."
from argos.robustness.feature_value import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm5-44-feature-family-rolling-revalidation-v1'

In [2]:
from argos.robustness.feature_value import build_feature_family_revalidation

result = build_feature_family_revalidation(
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    robustness_spec=M5_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
)
result.metrics


,model,split,rows,positives,prevalence,pr_auc,roc_auc,log_loss,brier,ece,family,evaluation_year,feature_count
0,contract_fixed_budget,feature_value_2022,710534,9289,0.013073,0.063352,0.750704,0.244906,0.067774,0.169949,contract,2022,23
1,contract_historical_fixed_budget,feature_value_2022,710534,9289,0.013073,0.069095,0.774356,0.228317,0.061716,0.158953,contract_historical,2022,35
2,all_fixed_budget,feature_value_2022,710534,9289,0.013073,0.069191,0.776313,0.226868,0.061110,0.158135,all,2022,41
3,contract_fixed_budget,feature_value_2023,843059,15283,0.018128,0.071542,0.769334,0.291956,0.086503,0.192705,contract,2023,23
4,contract_historical_fixed_budget,feature_value_2023,843059,15283,0.018128,0.090021,0.802681,0.261343,0.074992,0.172308,contract_historical,2023,35
5,all_fixed_budget,feature_value_2023,843059,15283,0.018128,0.089393,0.803728,0.263756,0.075955,0.173691,all,2023,41
6,contract_fixed_budget,feature_value_2024,950670,9180,0.009656,0.062608,0.816479,0.291702,0.088069,0.196885,contract,2024,23
7,contract_historical_fixed_budget,feature_value_2024,950670,9180,0.009656,0.054229,0.825819,0.268665,0.077498,0.178174,contract_historical,2024,35
8,all_fixed_budget,feature_value_2024,950670,9180,0.009656,0.055154,0.827292,0.267895,0.077205,0.177545,all,2024,41


In [3]:
result.deltas


,evaluation_year,family,reference_family,delta_pr_auc,delta_roc_auc,delta_lift_at_operational_k
0,2022,contract_historical,contract,0.005743,0.023652,0.753510
1,2022,all,contract,0.005839,0.025608,1.033384
2,2022,competition_increment,contract_historical,0.000096,0.001957,0.279875
3,2023,contract_historical,contract,0.018479,0.033347,2.748018
4,2023,all,contract,0.017852,0.034394,2.604074
5,2023,competition_increment,contract_historical,-0.000628,0.001047,-0.143944
6,2024,contract_historical,contract,-0.008379,0.009340,-0.305001
7,2024,all,contract,-0.007455,0.010813,0.239644
8,2024,competition_increment,contract_historical,0.000924,0.001473,0.544645


In [4]:
result.ranking


,model,split,review_fraction,reviewed,captured_positives,precision_at_k,recall_at_k,lift_at_k,cumulative_gain,family,evaluation_year
0,contract_fixed_budget,feature_value_2022,0.001,711,161,0.226442,0.017332,17.320969,0.017332,contract,2022
1,contract_fixed_budget,feature_value_2022,0.005,3553,560,0.157613,0.060286,12.056152,0.060286,contract,2022
2,contract_fixed_budget,feature_value_2022,0.010,7106,944,0.132845,0.101626,10.161614,0.101626,contract,2022
3,contract_fixed_budget,feature_value_2022,0.025,17764,1824,0.102680,0.196361,7.854164,0.196361,contract,2022
4,contract_fixed_budget,feature_value_2022,0.050,35527,2489,0.070059,0.267951,5.358982,0.267951,contract,2022
5,contract_fixed_budget,feature_value_2022,0.100,71054,3815,0.053692,0.410701,4.106974,0.410701,contract,2022
6,contract_historical_fixed_budget,feature_value_2022,0.001,711,179,0.251758,0.019270,19.257475,0.019270,contract_historical,2022
7,contract_historical_fixed_budget,feature_value_2022,0.005,3553,594,0.167183,0.063947,12.788133,0.063947,contract_historical,2022
8,contract_historical_fixed_budget,feature_value_2022,0.010,7106,1014,0.142696,0.109161,10.915123,0.109161,contract_historical,2022
9,contract_historical_fixed_budget,feature_value_2022,0.025,17764,1864,0.104931,0.200667,8.026405,0.200667,contract_historical,2022


In [5]:
result.validation


,check_id,category,blocking,passed,observed,requirement
0,feature_family_protocol_complete,ablation,True,True,"{""families"": [""all"", ""contract"", ""contract_his...",all frozen feature families are compared acros...
1,feature_family_same_budget,ablation,True,True,"{""max_iter"": 80, ""max_train_rows"": 200000, ""sa...","family comparisons use the same algorithm, row..."
2,feature_family_no_test_2025,governance,True,True,"{""consumed_test_year"": 2025, ""years"": [2022, 2...",feature-family value is revalidated without se...
3,competition_increment_measured,evidence,True,True,"{""competition_deltas"": [{""delta_lift_at_operat...",competition features have an explicit incremen...


In [6]:
blocking = result.validation.loc[result.validation["blocking"]]
print("44 ✅ PASS" if result.ready_for_next_notebook else "44 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook


44 ✅ PASS
Blocking checks: 4/4
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m5_feature_family_value.json
